# Section 7 — CNN regime 1 (Denoiser) · Compare Results
Loads the 5 methods' `results/*.json` from Drive (missing ones skipped). BCE + pixel-accuracy, cost/memory, three-factor cos-sweep head-to-head, and a sample-reconstruction grid.

## 1. Setup + Load Results

In [ ]:
import os, json, math, torch
import torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
device='cuda' if torch.cuda.is_available() else 'cpu'
USE_DRIVE, DRIVE_SUBDIR = True, 'Section7_r1_denoiser'
if USE_DRIVE:
    try:
        from google.colab import drive; drive.mount('/content/drive'); STORE=os.path.join('/content/drive/MyDrive',DRIVE_SUBDIR)
    except Exception as e:
        print('Drive mount failed:',e); STORE=os.path.join('/content',DRIVE_SUBDIR)
else:
    STORE=os.path.join('.',DRIVE_SUBDIR)
RESULTS_DIR=os.path.join(STORE,'results'); CKPT_DIR=os.path.join(STORE,'checkpoints')

# Pull in results/checkpoints that landed in ANOTHER Drive folder (searched AFTER this notebook's own folder).
# e.g. a run that saved to Section7_r1_autoencoder by mistake -> set the line below then re-run this cell:
#   EXTRA_RESULTS_DIRS = ['/content/drive/MyDrive/Section7_r1_autoencoder/results']
#   EXTRA_CKPT_DIRS    = ['/content/drive/MyDrive/Section7_r1_autoencoder/checkpoints']
EXTRA_RESULTS_DIRS = []
EXTRA_CKPT_DIRS    = []

# Clip the comparison FIGURES to this many hours (e.g. the 100M run went 5h but should show 2h). None = no clip.
CUTOFF_HOURS = 2.0
EXPECT_SIGMA = 0.5   # this is the DENOISER; loaded results should have CORRUPT_SIGMA=0.5 (warns if not -> likely an AE run)

METHODS=['backprop','two_factor','three_factor_clean','three_factor_normal','three_factor_noisy', 'backprop_100M']
LABELS={'backprop':'Backprop','two_factor':'Two-factor Hebbian','three_factor_clean':'Three-factor clean (cos~0.5)','three_factor_normal':'Three-factor normal (cos~0.09)','three_factor_noisy':'Three-factor noisy (cos~0.01)', 'backprop_100M':'Backprop 100M (ceiling)'}
COLORS={'backprop':'#1F3864','two_factor':'#B8860B','three_factor_clean':'#C62828','three_factor_normal':'#E67E22','three_factor_noisy':'#7B1FA2', 'backprop_100M':'#2E7D32'}

def _find(m):
    for d in [RESULTS_DIR]+EXTRA_RESULTS_DIRS:
        pth=os.path.join(d,f'{m}.json')
        if os.path.exists(pth): return pth
    return None
def _win(c):                                              # indices of a curve within CUTOFF_HOURS (None -> all)
    if CUTOFF_HOURS is None: return list(range(len(c['t_sec'])))
    return [i for i,t in enumerate(c['t_sec']) if t <= CUTOFF_HOURS*3600] or [0]

R={}
for m in METHODS:
    pth=_find(m)
    if pth:
        R[m]=json.load(open(pth)); s=R[m]['summary']; md=R[m]['meta']
        _sig=md.get('config',{}).get('CORRUPT_SIGMA')
        _w = '' if (_sig is None or abs(_sig-EXPECT_SIGMA)<1e-9) else f'   *** CORRUPT_SIGMA={_sig} (expected {EXPECT_SIGMA}: looks like an AUTOENCODER run, not a denoiser) ***'
        print(f'loaded {m:24} {md["total_steps"]:>9,} steps  {md["wall_clock_sec"]/3600:5.2f}h  best BCE {s["best_bce"]:.4f}  best pixel-acc {s["best_acc"]:.4f}{_w}')
    else:
        print(f'MISSING {m}.json  (run the {m} notebook, or add its folder to EXTRA_RESULTS_DIRS / upload it below)')

## 1b. Pull in a result from another Drive folder (optional)
If a run saved to a different folder — e.g. `three_factor_noisy.json` under `Section7_r1_autoencoder` — set `EXTRA_RESULTS_DIRS` in the load cell above and re-run it, or upload the JSON here. The load cell **warns** if a pulled result's `CORRUPT_SIGMA` isn't 0.5 (i.e. it's an autoencoder run, not a denoiser run).

In [ ]:
import os, shutil
try:
    from google.colab import files
    up = files.upload()                                   # pick <method>.json (and optionally <method>.pt)
    for fn in up:
        dst = '/content/extra_results' if fn.endswith('.json') else '/content/extra_ckpts'
        os.makedirs(dst, exist_ok=True); shutil.move(fn, os.path.join(dst, fn))
    if os.path.isdir('/content/extra_results') and '/content/extra_results' not in EXTRA_RESULTS_DIRS:
        EXTRA_RESULTS_DIRS.append('/content/extra_results')
    if os.path.isdir('/content/extra_ckpts') and '/content/extra_ckpts' not in EXTRA_CKPT_DIRS:
        EXTRA_CKPT_DIRS.append('/content/extra_ckpts')
    print('uploaded; RE-RUN the load cell above. EXTRA_RESULTS_DIRS =', EXTRA_RESULTS_DIRS)
except Exception as e:
    print('nothing uploaded (fine if everything is already on this Drive):', e)

## 2. Compare curves (BCE + pixel-acc)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
for m in R:
    c = R[m]['curve']; w=_win(c); hrs=[c['t_sec'][i]/3600 for i in w]
    ax[0].plot(hrs, [c['test_bce'][i] for i in w], 'o-', color=COLORS[m], label=LABELS[m])
    ax[1].plot(hrs, [c['test_acc'][i] for i in w], 'o-', color=COLORS[m], label=LABELS[m])
ax[0].set_xlabel('hours'); ax[0].set_ylabel('test BCE / pixel'); ax[0].set_title('Reconstruction BCE vs time'); ax[0].legend()
if R:
    bg = list(R.values())[0]['summary'].get('bg_baseline')
    if bg: ax[1].axhline(bg, color='gray', ls='--', lw=1, label=f'all-bg {bg:.2f}')
ax[1].set_xlabel('hours'); ax[1].set_ylabel('pixel accuracy'); ax[1].set_title('Pixel-accuracy vs time'); ax[1].legend()
if CUTOFF_HOURS:
    for a in ax: a.set_xlim(0, CUTOFF_HOURS)
    fig.suptitle(f'figures clipped to the first {CUTOFF_HOURS} h', y=1.02, fontsize=9, color='#666')
plt.tight_layout(); plt.show()

## 3. Cost & Memory

In [ ]:
def fwd_equiv(m):
    st = R[m]['meta']['total_steps']
    if m.startswith('three_factor'): return st * 2 * R[m]['meta']['config'].get('M', 0)
    if m.startswith('backprop'): return st * 3
    return st * 2
print(f'{"method":30}{"steps":>10}{"fwd-equiv":>14}{"wall h":>8}{"peak MB":>9}')
print('-'*71)
for m in R:
    md = R[m]['meta']
    print(f'{LABELS[m]:30}{md["total_steps"]:>10,}{fwd_equiv(m):>14,}{md["wall_clock_sec"]/3600:>8.2f}{md.get("peak_mem_mb", float("nan")):>9.1f}')

## Peak training memory — backprop vs. forward-only (no activation tape)

Measures the **activation tape** backprop must retain for its backward pass — memory the forward-only three-factor rule never allocates (it only *evaluates* the loss). Self-contained and unexecuted: run it to print the numbers next to the accuracy results. Exact (`torch.autograd.graph.saved_tensors_hooks`), at this section's training batch size.

In [ ]:
# ============================================================================
# Peak training memory: backprop vs the forward-only three-factor rule.
# Backprop must retain an "activation tape" (one saved tensor per layer op) so
# the backward pass can run; the forward-only rule only EVALUATES the loss, so
# it stores none of it -> training memory stays at the inference footprint.
# Measured exactly via autograd's own saved-tensor hooks (not an estimate).
# Also writes peak_training_memory.csv into the exports/ folder so it rides
# along in the results zip next to summary.csv. Self-contained + unexecuted.
# ============================================================================
import torch, torch.nn as nn, torch.nn.functional as F
_MEM_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

def _activation_tape_bytes(model, xshape, lossfn, make_target):
    model = model.to(_MEM_DEVICE)
    _pp = {p.untyped_storage().data_ptr() for p in model.parameters()}
    _seen, _isp = {}, {}
    def _pack(t):
        st = t.untyped_storage(); dp = st.data_ptr()
        _seen[dp] = st.nbytes(); _isp[dp] = (dp in _pp); return t
    def _unpack(t): return t
    x = torch.randn(*xshape, device=_MEM_DEVICE)
    with torch.autograd.graph.saved_tensors_hooks(_pack, _unpack):
        out = model(x); loss = lossfn(out, make_target(out)); loss.backward()
    return sum(nb for dp, nb in _seen.items() if not _isp[dp])  # activations only (params excluded)

def _report_training_memory(label, model, xshape, lossfn, make_target):
    P = sum(p.numel() for p in model.parameters()); pmem = P*4
    tape = _activation_tape_bytes(model, xshape, lossfn, make_target)
    common = 4*pmem; bp, loc = common + tape, common; MB = 1e6
    print("="*78)
    print(f"PEAK TRAINING MEMORY  |  {label}")
    print(f"  params P = {P:,}  ({pmem/MB:.2f} MB weights)   |   batch = {xshape[0]}   |   device = {_MEM_DEVICE}")
    print("-"*78)
    print(f"  activation tape backprop must retain : {tape/MB:8.2f} MB   ({tape/max(pmem,1):.1f}x the weights)")
    print(f"  forward-only three-factor rule       : {0.0:8.2f} MB   (no tape -- structural)")
    print(f"  est. training memory (same Adam both = 4P shared):")
    print(f"      backprop     : {bp/MB:8.2f} MB")
    print(f"      forward-only : {loc/MB:8.2f} MB   ->  backprop needs {bp/max(loc,1):.2f}x the memory")
    print(f"  (tape ~ linear in batch; measured via torch.autograd saved_tensors_hooks)")
    print("="*78)
    return {"section": label, "arch": type(model).__name__.replace("_Mem",""),
            "params": P, "batch": xshape[0], "weights_MB": round(pmem/MB, 2),
            "activation_tape_MB": round(tape/MB, 2), "tape_over_weights": round(tape/max(pmem,1), 2),
            "backprop_train_MB": round(bp/MB, 2), "forward_only_train_MB": round(loc/MB, 2),
            "backprop_over_forward_only": round(bp/max(loc,1), 2)}

class _MemConvAE(nn.Module):
    def __init__(self, conv_c=48, latent=98):
        super().__init__(); c2=2*conv_c; self._c2=c2
        self.enc=nn.Sequential(nn.Conv2d(1,conv_c,3,2,1),nn.ReLU(),nn.Conv2d(conv_c,c2,3,2,1),nn.ReLU())
        self.fc1=nn.Linear(c2*7*7, latent); self.fc2=nn.Linear(latent, c2*7*7)
        self.dec=nn.Sequential(nn.Upsample(scale_factor=2,mode='nearest'),nn.Conv2d(c2,conv_c,3,padding=1),nn.ReLU(),nn.Upsample(scale_factor=2,mode='nearest'),nn.Conv2d(conv_c,1,3,padding=1))
    def forward(self,x):
        h=self.enc(x); b=h.shape[0]; z=self.fc1(h.reshape(b,-1))
        h2=self.fc2(z).reshape(b,self._c2,7,7); return self.dec(h2)
_MEM_ROW = _report_training_memory("S7 CNN denoiser (MNIST, binarized)", _MemConvAE(), (128,1,28,28), lambda o,t: F.binary_cross_entropy_with_logits(o,t), lambda o:(torch.rand_like(o)>0.5).float())

# --- stash + persist so the export cell's zip picks it up (rides next to summary.csv) ---
PEAK_MEM_ROWS = [_MEM_ROW]
try:
    import os as _os, csv as _csv
    if 'STORE' in globals():
        _ed = _os.path.join(STORE, 'exports'); _os.makedirs(_ed, exist_ok=True)
        with open(_os.path.join(_ed, 'peak_training_memory.csv'), 'w', newline='') as _f:
            _w = _csv.DictWriter(_f, fieldnames=list(PEAK_MEM_ROWS[0].keys())); _w.writeheader()
            for _r in PEAK_MEM_ROWS: _w.writerow(_r)
        print('  saved ->', _os.path.join(_ed, 'peak_training_memory.csv'))
    else:
        print('  (STORE not defined yet -- CSV will be written by the export cell / re-run after Setup)')
except Exception as _e:
    print('  (peak_training_memory.csv not written:', _e, ')')


## 4. Head-to-head — three-factor cos sweep

In [ ]:
tf = [m for m in ['three_factor_clean','three_factor_normal','three_factor_noisy'] if m in R]
if len(tf) >= 2:
    fig, ax = plt.subplots(1, 3, figsize=(16, 4.3))
    for m in tf:
        c = R[m]['curve']; w=_win(c); hrs=[c['t_sec'][i]/3600 for i in w]
        ax[0].plot(hrs, [c['test_bce'][i] for i in w], 'o-', color=COLORS[m], label=LABELS[m])
        ax[1].plot(hrs, [c['test_acc'][i] for i in w], 'o-', color=COLORS[m], label=LABELS[m])
        ax[2].plot([c['step'][i] for i in w], [c['test_bce'][i] for i in w], 'o-', color=COLORS[m], label=LABELS[m])
    ax[0].set_xlabel('hours'); ax[0].set_ylabel('test BCE'); ax[0].set_title('cos sweep — BCE vs time'); ax[0].legend()
    ax[1].set_xlabel('hours'); ax[1].set_ylabel('pixel acc'); ax[1].set_title('pixel-acc vs time'); ax[1].legend()
    ax[2].set_xscale('symlog'); ax[2].set_xlabel('steps'); ax[2].set_ylabel('test BCE'); ax[2].set_title('BCE vs steps (within cutoff)'); ax[2].legend()
    if CUTOFF_HOURS:
        ax[0].set_xlim(0, CUTOFF_HOURS); ax[1].set_xlim(0, CUTOFF_HOURS)
    plt.tight_layout(); plt.show()
    print(f'{"variant":30}{"M":>10}{"cos~":>8}{"steps":>10}{"best_bce":>11}{"best_acc":>10}')
    print('-'*79)
    for m in tf:
        md, s = R[m]['meta'], R[m]['summary']; M = md['config'].get('M') or 0; P = md['P']
        cos = (M/(M+P+1))**0.5 if M else float('nan')
        print(f'{LABELS[m]:30}{M:>10,}{cos:>8.3f}{md["total_steps"]:>10,}{s["best_bce"]:>11.4f}{s["best_acc"]:>10.4f}')
    best = min(tf, key=lambda m: R[m]['summary']['best_bce'])
    print(f'\nLowest best BCE (full run): {LABELS[best]}.')
else:
    print('Head-to-head needs >=2 three-factor variants (03/04/05).')

## 5. Sample reconstructions (best model)

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

class ConvAE(nn.Module):
    """Conv encoder -> dense latent bottleneck -> Upsample+Conv decoder -> 28x28 logits.
    No BatchNorm (vmap-clean); Upsample+Conv instead of ConvTranspose (vmap-safe)."""
    def __init__(self, conv_c=48, latent=98):
        super().__init__()
        c2 = 2 * conv_c
        self.enc = nn.Sequential(
            nn.Conv2d(1, conv_c, 3, stride=2, padding=1), nn.ReLU(),   # 28 -> 14
            nn.Conv2d(conv_c, c2, 3, stride=2, padding=1), nn.ReLU(),  # 14 -> 7
        )
        self._c2 = c2
        self.fc1 = nn.Linear(c2 * 7 * 7, latent)
        self.fc2 = nn.Linear(latent, c2 * 7 * 7)
        self.dec = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='nearest'),               # 7 -> 14
            nn.Conv2d(c2, conv_c, 3, padding=1), nn.ReLU(),
            nn.Upsample(scale_factor=2, mode='nearest'),               # 14 -> 28
            nn.Conv2d(conv_c, 1, 3, padding=1),
        )
    def forward(self, x):
        h = self.enc(x)
        b = h.shape[0]
        z = self.fc1(h.reshape(b, -1))
        h2 = self.fc2(z).reshape(b, self._c2, 7, 7)
        return self.dec(h2)                                            # (B,1,28,28) logits

# sample reconstructions for the best model that has a checkpoint: input (noisy) | target | reconstruction
import torch, torchvision
def _ckpt(m):
    for d in [CKPT_DIR]+EXTRA_CKPT_DIRS:
        pth=os.path.join(d,f'{m}.pt')
        if os.path.exists(pth): return pth
    return None
_avail=[m for m in R if _ckpt(m)]
if _avail:
    best = max(_avail, key=lambda m: R[m]['summary']['best_acc'])
    cfg = R[best]['meta']['config']
    net = ConvAE(cfg['CONV_C'], cfg['LATENT']).to(device)
    ck = torch.load(_ckpt(best), map_location=device)['method']
    net.load_state_dict(ck['net'] if 'net' in ck else ck['params']); net.eval()
    _te = torchvision.datasets.MNIST('./data', train=False, download=True)
    x = (_te.data[:8].float()/255.0 > 0.5).float().unsqueeze(1).to(device)
    torch.manual_seed(cfg['SEED']); xin = x + cfg['CORRUPT_SIGMA']*torch.randn_like(x) if cfg['CORRUPT_SIGMA']>0 else x
    with torch.no_grad(): rec = (torch.sigmoid(net(xin)) > 0.5).float()
    fig, ax = plt.subplots(3, 8, figsize=(12, 4.6))
    for j in range(8):
        ax[0,j].imshow(xin[j,0].cpu(), cmap='gray'); ax[1,j].imshow(x[j,0].cpu(), cmap='gray'); ax[2,j].imshow(rec[j,0].cpu(), cmap='gray')
        for r in range(3): ax[r,j].axis('off')
    ax[0,0].set_ylabel('input'); ax[1,0].set_ylabel('target'); ax[2,0].set_ylabel('reconstruction')
    plt.suptitle(f'Best model: {LABELS[best]} (pixel-acc {R[best]["summary"]["best_acc"]:.3f})'); plt.tight_layout(); plt.show()
else:
    print('no checkpoints available for the reconstruction grid (run a method, or add EXTRA_CKPT_DIRS)')

## 6. Summary table

In [ ]:
print(f'{"Experiment":30}{"Steps":>10}{"Init BCE":>10}{"Best BCE":>10}{"Final acc":>11}{"Best acc":>10}')
print('-'*81)
for m in R:
    md, s = R[m]['meta'], R[m]['summary']
    print(f'{LABELS[m]:30}{md["total_steps"]:>10,}{s["initial_bce"]:>10.4f}{s["best_bce"]:>10.4f}{s["final_acc"]:>11.4f}{s["best_acc"]:>10.4f}')
print(f'\n(all-background pixel-acc baseline = {list(R.values())[0]["summary"]["bg_baseline"]:.3f} — methods must beat this)' if R else '')

## 7. Export figures + CSV → Drive

In [ ]:
# Export every figure (PNG) + data (CSV) -> a Drive folder, then download a zip.
import os, csv, glob, shutil
EXPORT_DIR = os.path.join(STORE, 'exports'); os.makedirs(EXPORT_DIR, exist_ok=True)
if R:
    ckeys=[]
    for m in R:
        for k in R[m]['curve']:
            if k not in ckeys: ckeys.append(k)
    with open(os.path.join(EXPORT_DIR,'curves.csv'),'w',newline='') as f:
        w=csv.writer(f); w.writerow(['method']+ckeys)
        for m in R:
            c=R[m]['curve']; n=len(c.get('t_sec',[]))
            for i in range(n): w.writerow([m]+[c[k][i] if (k in c and i<len(c[k])) else '' for k in ckeys])
    skeys=[]
    for m in R:
        for k in R[m]['summary']:
            if k not in skeys: skeys.append(k)
    with open(os.path.join(EXPORT_DIR,'summary.csv'),'w',newline='') as f:
        w=csv.writer(f); w.writerow(['method','P','total_steps','wall_clock_h','M','cos']+skeys)
        for m in R:
            md,s=R[m]['meta'],R[m]['summary']; M=md['config'].get('M') or 0
            cos=round((M/(M+md['P']+1))**0.5,4) if M else ''
            w.writerow([m,md['P'],md['total_steps'],round(md['wall_clock_sec']/3600,3),M,cos]+[s.get(k,'') for k in skeys])
    print('wrote curves.csv, summary.csv')
def _save(fig,name): fig.savefig(os.path.join(EXPORT_DIR,name),dpi=130,bbox_inches='tight'); plt.close(fig)
if R:
    fig, ax = plt.subplots(1,2,figsize=(12,4.5))
    for m in R:
        c=R[m]['curve']; ww=_win(c); hrs=[c['t_sec'][i]/3600 for i in ww]
        ax[0].plot(hrs,[c['test_bce'][i] for i in ww],'o-',color=COLORS[m],label=LABELS[m])
        ax[1].plot(hrs,[c['test_acc'][i] for i in ww],'o-',color=COLORS[m],label=LABELS[m])
    ax[0].set_xlabel('hours'); ax[0].set_ylabel('test BCE'); ax[0].set_title('Reconstruction BCE vs time'); ax[0].legend(fontsize=8)
    bg=list(R.values())[0]['summary'].get('bg_baseline')
    if bg: ax[1].axhline(bg,color='gray',ls='--',lw=1,label=f'all-bg {bg:.2f}')
    ax[1].set_xlabel('hours'); ax[1].set_ylabel('pixel accuracy'); ax[1].set_title('Pixel-accuracy vs time'); ax[1].legend(fontsize=8)
    if CUTOFF_HOURS:
        for a in ax: a.set_xlim(0,CUTOFF_HOURS)
    _save(fig,'curves_bce_pixacc.png'); print('wrote comparison figure')
for d in {RESULTS_DIR, CKPT_DIR, STORE}:
    for png in glob.glob(os.path.join(d,'*.png')):
        try: shutil.copy(png, EXPORT_DIR)
        except Exception: pass
print('\nexport folder:', EXPORT_DIR); print(' ', sorted(os.listdir(EXPORT_DIR)))
_zt = '/content' if os.path.isdir('/content') else os.path.dirname(EXPORT_DIR)
zip_path = shutil.make_archive(os.path.join(_zt, DRIVE_SUBDIR+'_exports'), 'zip', EXPORT_DIR)
try:
    if os.path.abspath(os.path.dirname(zip_path)) != os.path.abspath(STORE): shutil.copy(zip_path, STORE)
except Exception as e: print('(could not copy zip to Drive):', e)
print('zip:', zip_path)
try:
    from google.colab import files; files.download(zip_path)
except Exception as e:
    print('(download only runs in Colab):', e)